# PW-4: OpenF1 data spike

Explores what OpenF1 really returns for one full race, the **2026 Singapore Grand Prix**
(11 October 2026), before the ingestion service is designed.

Questions answered here:

1. How do meetings and sessions identify a race?
2. For each endpoint: rows, columns, types, null rates and quirks.
3. How many requests does one race cost under the free tier (3/s, 30/min)?
4. Can FastF1 fill gaps in OpenF1?

Findings are written up in `docs/spike-openf1.md`. This notebook is scratch work: no tests,
outputs stripped before commit (nbstripout).

Run with: `uv sync --group notebook`, then open in VS Code or `uv run jupyter lab`.

In [ ]:
import time
from collections import deque
from typing import Any

import httpx
import pandas as pd

BASE_URL = "https://api.openf1.org/v1"
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## A small rate limiter

The free tier allows 3 requests per second and 30 per minute. Keeping exactly 3 per second
on our side still produced HTTP 429 during the spike (the server's window and ours do not line
up), so this limiter keeps a margin: at most 2 per second and 28 per minute. It is a sliding
window: remember when each request was sent and wait until the oldest one leaves the window.

In [ ]:
class RateLimiter:
    """Blocks until a request may be sent without breaking either window."""

    def __init__(self, per_second: int = 2, per_minute: int = 28) -> None:
        self.limits = [(1.0, per_second), (60.0, per_minute)]
        self.sent: deque[float] = deque()

    def wait(self) -> None:
        while True:
            now = time.monotonic()
            while self.sent and now - self.sent[0] > 60.0:
                self.sent.popleft()
            delays = []
            for window, limit in self.limits:
                in_window = [t for t in self.sent if now - t < window]
                if len(in_window) >= limit:
                    delays.append(window - (now - in_window[0]) + 0.01)
            if not delays:
                self.sent.append(now)
                return
            time.sleep(max(delays))


limiter = RateLimiter()
calls: list[dict[str, Any]] = []


def get(client: httpx.Client, endpoint: str, **params: Any) -> list[dict[str, Any]]:
    """GET one endpoint, respecting the limiter and retrying once on 429."""
    for _ in range(2):
        limiter.wait()
        started = time.perf_counter()
        response = client.get(f"{BASE_URL}/{endpoint}", params=params)
        calls.append(
            {
                "endpoint": endpoint,
                "status": response.status_code,
                "seconds": round(time.perf_counter() - started, 2),
                "kb": round(len(response.content) / 1024, 1),
            }
        )
        if response.status_code == 429:
            time.sleep(float(response.headers.get("retry-after", "1")))
            continue
        if response.status_code == 404:  # OpenF1 answers 404 for "no rows"
            return []
        response.raise_for_status()
        return response.json()
    raise RuntimeError(f"{endpoint}: still rate limited after a retry")


client = httpx.Client(timeout=60)

## 1. Find the race: meeting and session keys

In [ ]:
meetings = pd.DataFrame(get(client, "meetings", year=2026, country_name="Singapore"))
meeting_key = int(meetings.loc[0, "meeting_key"])
sessions = pd.DataFrame(get(client, "sessions", meeting_key=meeting_key))
race_key = int(sessions.loc[sessions.session_name == "Race", "session_key"].iloc[0])
quali_key = int(sessions.loc[sessions.session_name == "Qualifying", "session_key"].iloc[0])
print(f"meeting_key={meeting_key} race session_key={race_key} qualifying session_key={quali_key}")
sessions[["session_key", "session_name", "session_type", "date_start", "date_end", "gmt_offset"]]

## 2. Pull every endpoint for the race

In [ ]:
ENDPOINTS = [
    "drivers",
    "laps",
    "stints",
    "pit",
    "weather",
    "race_control",
    "position",
    "intervals",
    "session_result",
    "overtakes",
]
started = time.perf_counter()
data = {name: pd.DataFrame(get(client, name, session_key=race_key)) for name in ENDPOINTS}
data["starting_grid"] = pd.DataFrame(get(client, "starting_grid", session_key=quali_key))
elapsed = time.perf_counter() - started
print(f"{len(ENDPOINTS) + 1} requests in {elapsed:.1f} s")
pd.DataFrame(calls)

## 3. Profile: rows, types and null rates per endpoint

In [ ]:
def profile(df: pd.DataFrame) -> pd.DataFrame:
    """One row per column: type, null rate and an example value."""
    return pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "null_rate": df.isna().mean().round(3),
            "example": [str(df[c].dropna().iloc[0])[:50] if df[c].notna().any() else None for c in df],
        }
    )


for name, df in data.items():
    print(f"\n=== {name}: {len(df)} rows")
    print(profile(df).to_string())

## 4. Laps: missing lap times, lap 1, pit laps

In [ ]:
laps, results = data["laps"], data["session_result"]
missing = laps[laps.lap_duration.isna()][["driver_number", "lap_number"]]
retired = results[results.dnf][["driver_number", "number_of_laps"]]
print("Laps without a lap time (all are the lap a driver retired on):")
print(missing.merge(retired, on="driver_number").to_string(index=False))

lap1 = laps[laps.lap_number == 1]
print(f"\nLap 1 date_start values: {lap1.date_start.nunique()} distinct (the start signal)")
print(f"Lap 1 median {lap1.lap_duration.median():.1f} s vs race median {laps.lap_duration.median():.1f} s")
print(f"Pit-out laps flagged: {laps.is_pit_out_lap.sum()}; pit-in laps have no flag")

## 5. Stints and pit stops

In [ ]:
stints, pit = data["stints"], data["pit"]
stints["laps"] = stints.lap_end - stints.lap_start + 1
coverage = stints.groupby("driver_number")["laps"].sum() - laps.groupby("driver_number").lap_number.max()
print("Drivers whose stints cover fewer laps than they started:", coverage[coverage != 0].to_dict())
print("Compounds:", stints.compound.value_counts().to_dict())
print("Stints started on used tyres:", int((stints.tyre_age_at_start > 0).sum()))
print(
    "Stint changes:",
    int((stints.groupby("driver_number").stint_number.max() - 1).sum()),
    "| pit rows:",
    len(pit),
)
print("pit_duration == lane_duration in every row:", bool((pit.pit_duration == pit.lane_duration).all()))
pit[["driver_number", "lap_number", "stop_duration", "lane_duration"]].describe().round(1)

## 6. Race control: safety car periods, flags

In [ ]:
rc = data["race_control"]
print(rc.category.value_counts().to_dict())
rc[rc.category == "SafetyCar"][["date", "lap_number", "message"]]

## 7. Weather, positions and gaps

In [ ]:
weather, intervals = data["weather"], data["intervals"]
step = pd.to_datetime(weather.date, format="ISO8601").diff().dt.total_seconds().median()
print(f"Weather every {step:.0f} s; rainfall max {weather.rainfall.max()}")
kinds = intervals.gap_to_leader.map(lambda v: type(v).__name__).value_counts().to_dict()
print("gap_to_leader value types:", kinds)
print(
    "Lapped values:",
    sorted(intervals.gap_to_leader[intervals.gap_to_leader.map(lambda v: isinstance(v, str))].unique()),
)
per_driver = pd.to_datetime(intervals.date, format="ISO8601").groupby(intervals.driver_number).diff()
print(
    f"Intervals: {len(intervals)} rows, one per driver every {per_driver.dt.total_seconds().median():.1f} s"
)

## 8. Request budget

Every race needs about 12 requests (sessions, the endpoints above, and the qualifying grid).
At 28 per minute, a full season (about 24 weekends, race plus qualifying plus sprints) is
roughly 350 to 400 requests, so about 15 minutes. Backfilling 2023 to 2026 takes about an hour.
`intervals` is the only large response (about 4 MB per race).

In [ ]:
pd.DataFrame(calls).groupby("endpoint")[["seconds", "kb"]].sum().sort_values("kb", ascending=False)

## 9. FastF1

`fastf1.get_session(2026, "Singapore", "R").load()` now logs
"Failed to authenticate with Formula 1 account" and loads no laps, weather or race control:
F1's timing archive needs a personal F1 account. That breaks the "free, no personal
credentials" rule, so FastF1 is dropped (ADR 3). Not run here, to keep the notebook
working without an account.

In [ ]:
client.close()